# Behavioral Momentum and Response Persistence

In this lab, you will analyze resistance to change across reinforcement contexts, using three different *disruptors*. Behavioral momentum theory (Nevin, 1992; Nevin & Shahan, 2011) predicts that responding maintained by richer reinforcement is more resistant to disruption.

The lab has two parts:

- **Part 1 - Prefeeding.** Fit the simplified momentum equation to multiple-schedule data where prefeeding is the disruptor, and compare resistance to change across rich and lean components.
- **Part 2 - Other disruptors.** Fit Nevin's resistance-to-extinction and alternative-reinforcement equations to data where extinction and alternative reinforcement are the disruptors.

## Background (Part 1)

Four subjects responded on a two-component multiple schedule with **rich** and **lean** reinforcement components. After baseline response rates were established, responding was disrupted by prefeeding at graded levels (0, 25, 50, 75, and 100g). The data record baseline and disrupted rates for each subject, component, and disruption level.

The behavioral momentum equation describes the relation between disruption and proportional change in responding:

$$\log\left(\frac{B_x}{B_0}\right) = \frac{-x \cdot c}{r \cdot S}$$

where $B_x$ is the response rate under disruption level $x$, $B_0$ is the baseline rate, $c$ is sensitivity to disruption, $r$ is the reinforcement rate, and $S$ captures the stimulus-reinforcer relation.

## Task 1: Import Libraries

Import the libraries you will need for this lab: `pandas`, `numpy`, `matplotlib.pyplot`, and `scipy.optimize`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Task 2: Load and Inspect the Data

Load `momentum_data.csv` into a DataFrame. Examine the structure of the data. How many subjects are there? What are the unique disruption levels? Print summary statistics for the rich and lean components separately.

In [ ]:
df = pd.read_csv("momentum_data.csv")

print("Number of subjects:", df['subject'].nunique())
print("Disruption levels:", sorted(df['disruption_level'].unique()))
print("Components:", df['component'].unique())

for comp in ['rich', 'lean']:
    print(f"\n--- {comp} component ---")
    print(df[df['component'] == comp][['baseline_rate', 'disrupted_rate']].describe())

df.head()

## Task 3: Calculate Proportion of Baseline

For each row in the dataset, calculate the proportion of baseline responding ($B_x / B_0$). Add this as a new column called `prop_baseline`. Then calculate $\log_{10}(B_x / B_0)$ and store it in a column called `log_prop_baseline`.

**Note:** At disruption level 0, the proportion should be 1.0 and the log proportion should be 0.0. Verify this is the case in your data.

In [ ]:
df['prop_baseline'] = df['disrupted_rate'] / df['baseline_rate']
df['log_prop_baseline'] = np.log10(df['prop_baseline'])

check = df[df['disruption_level'] == 0][['prop_baseline', 'log_prop_baseline']]
print("At disruption level 0:")
print("  prop_baseline values:", check['prop_baseline'].unique())
print("  log_prop_baseline values:", check['log_prop_baseline'].unique())

df.head()

## Task 4: Visualize the Raw Disruption Functions

Create a figure with two panels (subplots). In the left panel, plot proportion of baseline ($B_x / B_0$) as a function of disruption level, with separate lines for each subject. Use different colors or markers for rich vs. lean components. In the right panel, plot the same data on a log scale ($\log_{10}(B_x / B_0)$ on the y-axis).

Add appropriate axis labels, legends, and a title. What pattern do you notice regarding the rich vs. lean components?

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5))
colors = {'rich': 'tab:blue', 'lean': 'tab:orange'}
seen = set()

for (subj, comp), g in df.groupby(['subject', 'component']):
    g = g.sort_values('disruption_level')
    label = comp if comp not in seen else None
    seen.add(comp)
    axL.plot(g['disruption_level'], g['prop_baseline'], marker='o',
             color=colors[comp], alpha=0.6, label=label)
    axR.plot(g['disruption_level'], g['log_prop_baseline'], marker='o',
             color=colors[comp], alpha=0.6, label=label)

axL.set_xlabel('Disruption Level (g prefeeding)'); axL.set_ylabel('Bx / B0')
axL.set_title('Proportion of Baseline'); axL.legend()
axR.set_xlabel('Disruption Level (g prefeeding)'); axR.set_ylabel('log10(Bx / B0)')
axR.set_title('Log Proportion of Baseline'); axR.legend()
plt.tight_layout()
plt.show()

## Task 5: Compute Group Means

Calculate the mean proportion of baseline and mean log proportion of baseline across subjects for each combination of component (rich/lean) and disruption level. Store these in a new DataFrame. This will be useful for fitting the momentum equation to averaged data.

In [ ]:
group_means = (df.groupby(['component', 'disruption_level'])
                 [['prop_baseline', 'log_prop_baseline']]
                 .mean()
                 .reset_index())
group_means

## Task 6: Define the Behavioral Momentum Equation

Write a Python function that implements the behavioral momentum equation:

$$\log\left(\frac{B_x}{B_0}\right) = \frac{-x \cdot c}{r \cdot S}$$

The function should take `x` (disruption level) as the independent variable and `c`, `r`, and `S` as parameters. It should return the predicted log proportion of baseline.

**Hint:** Since $r$ and $S$ always appear as the product $r \cdot S$, you may find it easier to define a combined parameter (e.g., `rS`) to avoid identifiability issues during fitting. Think about what this means for interpreting your results.

In [ ]:
def momentum_model(x, c, rS):
    """Behavioral momentum: log10(Bx/B0) = -(c * x) / (r * S).

    r and S enter only as the product r*S, so they are combined into a single
    rS parameter. Only the ratio c/rS (the slope) is identifiable from one
    disruption function -- see Task 7.
    """
    return -(c * x) / rS

## Task 7: Fit the Model to Each Component

Using `scipy.optimize.curve_fit`, fit the behavioral momentum equation to the group-mean data for the **rich** component and the **lean** component separately.

Report the best-fitting parameter values for each component. Which component shows a steeper disruption function (i.e., less resistance to change)?

**Hints:**
- Exclude the disruption level 0 data point from fitting (it is trivially 0 on the log scale).
- Provide reasonable initial parameter guesses.
- Consider whether parameter bounds are needed.

In [ ]:
# Only c/rS is identifiable, so we fix c = 1.0 and let rS absorb the slope.
# A larger rS means a flatter function -> more resistance to change.
C_FIXED = 1.0

def fit_one(x, rS):
    return momentum_model(x, C_FIXED, rS)

fits = {}
for comp in ['rich', 'lean']:
    gm = group_means[(group_means['component'] == comp) &
                     (group_means['disruption_level'] > 0)]
    x = gm['disruption_level'].values.astype(float)
    y = gm['log_prop_baseline'].values
    popt, _ = curve_fit(fit_one, x, y, p0=[100.0], bounds=(1e-6, np.inf))
    rS = popt[0]
    fits[comp] = {'c': C_FIXED, 'rS': rS, 'slope': C_FIXED / rS}
    print(f"{comp:>4}: c={C_FIXED}, rS={rS:.2f}, slope=c/rS={C_FIXED/rS:.5f}")

steeper = min(fits, key=lambda c: fits[c]['rS'])
print(f"\nSteeper (less resistant) component: {steeper}")

## Task 8: Evaluate Model Fit

For each component, calculate the following goodness-of-fit measures:

1. **R-squared ($R^2$):** proportion of variance accounted for.
2. **Root Mean Squared Error (RMSE):** average magnitude of the residuals.

How well does the behavioral momentum equation describe the disruption data? Are the fits comparable across components?

In [ ]:
for comp in ['rich', 'lean']:
    gm = group_means[(group_means['component'] == comp) &
                     (group_means['disruption_level'] > 0)]
    x = gm['disruption_level'].values.astype(float)
    y = gm['log_prop_baseline'].values
    pred = momentum_model(x, fits[comp]['c'], fits[comp]['rS'])
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot
    rmse = np.sqrt(np.mean((y - pred) ** 2))
    fits[comp]['r2'] = r2
    fits[comp]['rmse'] = rmse
    print(f"{comp:>4}: R2={r2:.3f}, RMSE={rmse:.4f}")

## Task 9: Plot Model Predictions Against Data

Create a publication-quality figure showing:
- The group-mean log proportion of baseline data points for each component (use distinct markers for rich and lean).
- The fitted model predictions as smooth curves overlaid on the data.
- A legend identifying the components.
- Appropriate axis labels ("Disruption Level (g prefeeding)" and "log(Bx/B0)").

Generate the smooth curves by evaluating the model function at many x-values between 0 and 100.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
markers = {'rich': 'o', 'lean': 's'}
colors = {'rich': 'tab:blue', 'lean': 'tab:orange'}
xx = np.linspace(0, 100, 200)

for comp in ['rich', 'lean']:
    gm = group_means[group_means['component'] == comp]
    ax.scatter(gm['disruption_level'], gm['log_prop_baseline'],
               marker=markers[comp], color=colors[comp], s=70,
               label=f"{comp} (data)", zorder=3)
    yy = momentum_model(xx, fits[comp]['c'], fits[comp]['rS'])
    ax.plot(xx, yy, color=colors[comp],
            label=f"{comp} (fit, rS={fits[comp]['rS']:.1f})")

ax.set_xlabel('Disruption Level (g prefeeding)')
ax.set_ylabel('log(Bx/B0)')
ax.set_title('Behavioral Momentum Model: Predictions vs. Data')
ax.legend()
plt.tight_layout()
plt.show()

## Task 10: Interpretation and Discussion (Part 1)

In a markdown cell below, address the following:

1. How do the fitted parameter values differ between the rich and lean components? What do these differences mean in terms of behavioral momentum theory?
2. Is resistance to change greater in the rich component, as predicted by the theory? How do you quantify this from your model fits?
3. What are the limitations of using the simplified momentum equation with prefeeding as the disruptor?
4. If you combined the parameter $r \cdot S$ into a single value, what additional information would you need to separate them? Why might this matter for applied predictions?

## Part 2: Other Disruptors

### Task 11: Resistance to Extinction

Prefeeding is only one way to disrupt responding. Load `behavioral_momentum_extinction_data.csv`, where **extinction** is the disruptor across sessions. Implement Nevin's resistance-to-extinction equation

$$\frac{B_t}{B_0} = 10^{\,-t\,(c + d\,r)/\sqrt{r}}$$

and use `curve_fit` to estimate `c`, `d`, `r`, and `B0` for each condition. Report the fitted parameters and $R^2$, and overlay the fitted curves on the data. (Bounds: $c, d > 0$; $r > 0.1$; $B_0$ a plausible baseline rate.)

In [ ]:
ext = pd.read_csv('behavioral_momentum_extinction_data.csv')

def momentum_extinction(t, c, d, r, B0):
    """Resistance to extinction: Bt/B0 = 10^(-t * (c + d*r) / sqrt(r))."""
    return B0 * 10 ** ((-t * (c + d * r)) / (r ** 0.5))

ext_results = []
for cond in ext['condition'].unique():
    cd = ext[ext['condition'] == cond]
    t = cd['session'].values.astype(float)
    B = cd['response_rate'].values
    popt, _ = curve_fit(momentum_extinction, t, B, p0=[0.2, 0.1, 1.0, 40.0],
                        bounds=([0, 0, 0.1, 0], [1, 1, 5, 100]))
    pred = momentum_extinction(t, *popt)
    r2 = 1 - np.sum((B - pred) ** 2) / np.sum((B - np.mean(B)) ** 2)
    ext_results.append({'condition': cond, 'c': popt[0], 'd': popt[1],
                        'r': popt[2], 'B0': popt[3], 'r2': r2})
    print(f"{cond}: c={popt[0]:.3f}, d={popt[1]:.3f}, r={popt[2]:.3f}, "
          f"B0={popt[3]:.1f}, R2={r2:.3f}")

fig, ax = plt.subplots(figsize=(9, 5))
for row in ext_results:
    cd = ext[ext['condition'] == row['condition']].sort_values('session')
    ax.scatter(cd['session'], cd['response_rate'], s=30)
    tt = np.linspace(cd['session'].min(), cd['session'].max(), 100)
    ax.plot(tt, momentum_extinction(tt, row['c'], row['d'], row['r'], row['B0']),
            label=row['condition'])
ax.set_xlabel('Extinction session'); ax.set_ylabel('Response rate')
ax.set_title('Resistance to Extinction'); ax.legend()
plt.tight_layout(); plt.show()

### Task 12: Disruption by Alternative Reinforcement

Load `behavioral_momentum_alternative_data.csv`, where responding is disrupted by **alternative reinforcement** delivered at rate $R_a$. Implement

$$\frac{B_x}{B_0} = 10^{\,-p\,R_a/\sqrt{r + R_a}}$$

and fit `p`, `r`, and `B0` per condition. Overlay the fitted curves, and quantify how much responding is suppressed from baseline to the highest alternative-reinforcement rate.

In [ ]:
alt = pd.read_csv('behavioral_momentum_alternative_data.csv')

def momentum_alternative(Ra, p, r, B0):
    """Disruption by alternative reinforcement: Bx/B0 = 10^(-p*Ra / sqrt(r + Ra))."""
    return B0 * 10 ** ((-p * Ra) / ((r + Ra) ** 0.5))

alt_results = []
for cond in alt['condition'].unique():
    cd = alt[alt['condition'] == cond]
    Ra = cd['Ra'].values.astype(float)
    B = cd['response_rate'].values
    popt, _ = curve_fit(momentum_alternative, Ra, B, p0=[0.3, 1.0, 50.0],
                        bounds=([0, 0.1, 0], [2, 5, 100]))
    pred = momentum_alternative(Ra, *popt)
    r2 = 1 - np.sum((B - pred) ** 2) / np.sum((B - np.mean(B)) ** 2)
    baseline = cd[cd['Ra'] == 0]['response_rate'].iloc[0]
    final = cd[cd['Ra'] == cd['Ra'].max()]['response_rate'].iloc[0]
    suppression = (baseline - final) / baseline * 100
    alt_results.append({'condition': cond, 'p': popt[0], 'r': popt[1],
                        'B0': popt[2], 'r2': r2, 'suppression_pct': suppression})
    print(f"{cond}: p={popt[0]:.3f}, r={popt[1]:.3f}, B0={popt[2]:.1f}, "
          f"R2={r2:.3f}, suppression={suppression:.1f}%")

fig, ax = plt.subplots(figsize=(9, 5))
for row in alt_results:
    cd = alt[alt['condition'] == row['condition']].sort_values('Ra')
    ax.scatter(cd['Ra'], cd['response_rate'], s=30)
    rr = np.linspace(cd['Ra'].min(), cd['Ra'].max(), 100)
    ax.plot(rr, momentum_alternative(rr, row['p'], row['r'], row['B0']),
            label=row['condition'])
ax.set_xlabel('Alternative reinforcement rate (Ra)'); ax.set_ylabel('Response rate')
ax.set_title('Disruption by Alternative Reinforcement'); ax.legend()
plt.tight_layout(); plt.show()

## Wrap-up

Across all three disruptors (prefeeding, extinction, alternative reinforcement), what is the common signature of behavioral momentum? Compare how each disruptor enters its equation. Which parameters index *sensitivity to disruption* versus the *baseline mass* of behavior, and how do the recovered values line up with the theory's prediction that richer reinforcement yields greater resistance to change?